In [1]:
%pip install -q pandas requests pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 1. Configuração IAedu e funções partilhadas

In [2]:
import os, json, uuid, re, math
from pathlib import Path
from typing import Literal
import pandas as pd
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ConfigDict, ValidationError

# Procura .env na pasta do notebook ou em pastas superiores.
def find_project_root():
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        if (folder / ".env").is_file():
            return folder
    return cwd

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env", override=True)
API_ENDPOINT = os.getenv("IAEDU_API_ENDPOINT", "").strip()
CHANNEL_ID = os.getenv("IAEDU_CHANNEL_ID", "").strip()
API_KEY = os.getenv("IAEDU_API_KEY", "").strip()
if not all((API_ENDPOINT, CHANNEL_ID, API_KEY)):
    raise RuntimeError("Configuração IAedu em falta. Cria .env com IAEDU_API_ENDPOINT, IAEDU_CHANNEL_ID e IAEDU_API_KEY.")
if not API_ENDPOINT.startswith("https://"):
    raise ValueError("Usa o endpoint IAedu HTTPS completo, terminado em /stream (conforme o teu serviço).")
print("IAedu carregada do .env (credenciais ocultas).")

def locate_csv(filename):
    paths = [PROJECT_ROOT / "processed_data" / filename,
             PROJECT_ROOT / "data" / "processed_data" / filename,
             PROJECT_ROOT / "data" / filename,
             Path.cwd() / "processed_data" / filename,
             Path.cwd() / "data" / "processed_data" / filename]
    for path in paths:
        if path.is_file(): return path
    raise FileNotFoundError(f"Não encontrei {filename}. Procurei: {[str(p) for p in paths]}")


def clean_scalar(value):
    if value is None:
        return None

    if isinstance(value, str):
        value = value.strip()
        return value if value else None

    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        value = value.item()

    if isinstance(value, float) and not math.isfinite(value):
        return None

    return value


def parse_list_field(value):
    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if item is not None and str(item).strip()
        ]

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        item.strip()
        for item in value.split(";")
        if item.strip()
    ]

IAedu carregada do .env (credenciais ocultas).


### 2. Carregamento e verificação dos dados

In [3]:
CLINICAL_DATA_PATH = locate_csv("clinical_agent_data.csv")
clinical_df = pd.read_csv(CLINICAL_DATA_PATH)
CLINICAL_COLUMNS = ("age","gender","unit_type","admission_source","admission_diagnosis","diagnoses_0_24h","past_history")

required_columns = {"patientunitstayid",*CLINICAL_COLUMNS}

missing_columns = (required_columns- set(clinical_df.columns))

if missing_columns:
    raise ValueError("Colunas em falta: "
        f"{sorted(missing_columns)}")

if clinical_df["patientunitstayid"].duplicated().any():

    raise ValueError(
        "Há patientunitstayid duplicados "
        "no clinical_agent_data.csv."
    )

clinical_df = clinical_df.set_index("patientunitstayid",drop=False)

print("Clinical dataset:",clinical_df.shape, "|",CLINICAL_DATA_PATH)

Clinical dataset: (2520, 8) | C:\Users\ASUS\Desktop\1º ano m\IACH\projeto\eicu-collaborative-research-database-demo-2.0.1\processed_data\clinical_agent_data.csv


### 3. Contratos de saída estruturada

In [4]:
class ClinicalFinding(BaseModel):

    model_config = ConfigDict(
        extra="forbid"
    )

    category: Literal[
        "age",
        "gender",
        "unit_type",
        "admission_source",
        "admission_diagnosis",
        "diagnosis_0_24h",
        "past_history"
    ]

    value: str
    
class ClinicalAgentOutput(BaseModel):

    model_config = ConfigDict(
        extra="forbid"
    )

    patient_id: int

    status: Literal[
        "success",
        "insufficient_data",
        "error"
    ]

    findings: list[
        ClinicalFinding
    ] = Field(
        default_factory=list
    )
    
class ClinicalRunState(BaseModel):

    patient_id: int

    status: Literal[
        "success",
        "insufficient_data",
        "error",
        "running"
    ]

    raw_data: dict = Field(
        default_factory=dict
    )

    evidence: list[dict] = Field(
        default_factory=list
    )

    agent_output: (
        ClinicalAgentOutput | None
    ) = None

    error_message: (
        str | None
    ) = None
    


### 4. Ferramentas determinísticas

In [5]:
def get_clinical_data(
    patientunitstayid: int
) -> dict:
    """
    Consulta os dados clínicos estruturados
    de um internamento na ICU.
    """

    if patientunitstayid not in clinical_df.index:

        return {
            "error":
            f"Patient {patientunitstayid} not found."
        }


    row = clinical_df.loc[
        patientunitstayid
    ]


    return {
        "patientunitstayid":
            int(patientunitstayid),

        "age":
            clean_scalar(
                row["age"]
            ),

        "gender":
            clean_scalar(
                row["gender"]
            ),

        "unit_type":
            clean_scalar(
                row["unit_type"]
            ),

        "admission_source":
            clean_scalar(
                row["admission_source"]
            ),

        "admission_diagnosis":
            clean_scalar(
                row["admission_diagnosis"]
            ),

        "diagnoses_0_24h":
            parse_list_field(
                row["diagnoses_0_24h"]
            ),

        "past_history":
            parse_list_field(
                row["past_history"]
            )
    }
    
def build_clinical_evidence(
    patient_data: dict
) -> list[dict]:
    """
    Builds the exact set of clinical facts that
    the LLM is allowed to select as findings.
    """

    evidence = []


    age = patient_data.get("age")

    if age is not None:

        # Avoid displaying integer ages as 63.0
        if isinstance(age, float) and age.is_integer():
            age = int(age)

        evidence.append({
            "category": "age",
            "value": str(age)
        })


    gender = patient_data.get("gender")

    if gender:

        evidence.append({
            "category": "gender",
            "value": str(gender)
        })


    unit_type = patient_data.get(
        "unit_type"
    )

    if unit_type:

        evidence.append({
            "category": "unit_type",
            "value": str(unit_type)
        })


    admission_source = patient_data.get(
        "admission_source"
    )

    if admission_source:

        evidence.append({
            "category": "admission_source",
            "value": str(admission_source)
        })


    admission_diagnosis = patient_data.get(
        "admission_diagnosis"
    )

    if admission_diagnosis:

        evidence.append({
            "category": "admission_diagnosis",
            "value": str(admission_diagnosis)
        })


    for diagnosis in patient_data.get(
        "diagnoses_0_24h",
        []
    ):

        evidence.append({
            "category": "diagnosis_0_24h",
            "value": str(diagnosis)
        })


    for history in patient_data.get(
        "past_history",
        []
    ):

        evidence.append({
            "category": "past_history",
            "value": str(history)
        })


    return evidence

### 5. Cliente IAedu e validação das respostas

In [6]:
# IAedu comunica por eventos de streaming (SSE).
# Não enviar informações identificáveis.

def ask_iaedu(message: str) -> str:
    """Envia uma mensagem à IAedu e interpreta o streaming."""

    headers = {"x-api-key": API_KEY}

    fields = {
        "channel_id": (None, CHANNEL_ID),
        "thread_id": (None, str(uuid.uuid4())),
        "user_info": (None, "{}"),
        "message": (None, message),
    }

    chunks = []
    final_message = None

    with requests.post(
        API_ENDPOINT,
        headers=headers,
        files=fields,
        stream=True,
        timeout=(15, 120)
    ) as response:

        response.raise_for_status()

        for raw in response.iter_lines(decode_unicode=True):

            if not raw:
                continue

            line = (
                raw if isinstance(raw, str)
                else raw.decode("utf-8", errors="replace")
            ).strip()

            # Ignorar metadados SSE
            if line.startswith(":") or line.startswith("event:"):
                continue

            # Aceitar JSON direto e SSE com prefixo data:
            if line.startswith("data:"):
                line = line[5:].strip()

            if line == "[DONE]":
                break

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            if not isinstance(obj, dict):
                continue

            event_type = obj.get("type")

            if event_type == "error":
                raise RuntimeError(
                    f"IAedu error: {str(obj.get('content', obj.get('message', 'unknown')))[:300]}"
                )

            elif event_type == "token":
                content = obj.get("content")

                if isinstance(content, str):
                    chunks.append(content)

            elif event_type == "message":
                content = obj.get("content")

                if isinstance(content, dict):
                    text = content.get("content")

                    if isinstance(text, str):
                        final_message = text

            elif isinstance(obj.get("choices"), list):
                for choice in obj["choices"]:
                    delta = choice.get("delta", {})
                    if isinstance(delta, dict):
                        text = delta.get("content")
                        if isinstance(text, str):
                            chunks.append(text)

    answer = (
        final_message
        if final_message is not None
        else "".join(chunks)
    ).strip()

    if not answer:
        raise ValueError(
            "A IAedu devolveu uma resposta vazia "
            "ou um formato de streaming não reconhecido."
        )
    return answer

def parse_agent_json(
    reply: str,
    output_model
):

    content = reply.strip()


    if content.startswith("```"):

        content = re.sub(
            r"^```(?:json)?\s*",
            "",
            content,
            flags=re.I
        )

        content = re.sub(
            r"\s*```$",
            "",
            content
        ).strip()


    return output_model.model_validate_json(
        content
    )

In [7]:
def validated_selection(
    output,
    patient_id,
    evidence
):

    if (
        output.patient_id != patient_id
        or output.status == "error"
    ):

        raise ValueError(
            "ID ou estado devolvido "
            "pelo LLM inválido."
        )


    allowed = {
        (
            row["category"],
            row["value"]
        )
        for row in evidence
    }


    selected = set()


    for finding in output.findings:

        key = (
            finding.category,
            finding.value
        )


        if key in selected:

            raise ValueError(
                f"Finding duplicado: {key}"
            )


        selected.add(key)


        if key not in allowed:

            raise ValueError(
                "Finding não suportado "
                f"pelos dados: {key}"
            )


    if (
        output.status == "insufficient_data"
        and output.findings
    ):

        raise ValueError(
            "insufficient_data não pode "
            "conter findings."
        )

### 6. Execução do agente

In [8]:
def run_clinical_agent(
    patient_id: int
) -> ClinicalRunState:


    patient = get_clinical_data(
        patient_id
    )


    if "error" in patient:

        return ClinicalRunState(
            patient_id=patient_id,
            status="error",

            error_message=patient[
                "error"
            ],

            agent_output=(
                ClinicalAgentOutput(
                    patient_id=patient_id,
                    status="error",
                    findings=[]
                )
            )
        )


    evidence = build_clinical_evidence(
        patient
    )


    state = ClinicalRunState(
        patient_id=patient_id,
        status="running",
        raw_data=patient,
        evidence=evidence
    )


    if not evidence:

        state.status = (
            "insufficient_data"
        )

        state.agent_output = (
            ClinicalAgentOutput(
                patient_id=patient_id,
                status="insufficient_data",
                findings=[]
            )
        )

        return state


    prompt = f"""
You are the Clinical Agent in a de-identified ICU research prototype.

Only select entries from the deterministic PROVIDED EVIDENCE.

Do not:
- invent observations;
- modify or rephrase evidence;
- add diagnoses;
- infer diagnoses that are not explicitly present;
- calculate a severity score;
- predict mortality;
- recommend treatment;
- provide medical advice.

Your role is only to select the most informative clinical facts already
present in the evidence.

Give priority to:
1. admission diagnosis;
2. diagnoses documented during the first 24 hours;
3. relevant past medical history.

Age, gender, ICU type and admission source may also be selected when useful
for describing the patient's clinical context.

Every finding must correspond EXACTLY to one item in PROVIDED EVIDENCE.

Return ONLY a JSON object with exactly:
patient_id, status, findings

Each finding must contain exactly:
category, value

Allowed categories:
age
gender
unit_type
admission_source
admission_diagnosis
diagnosis_0_24h
past_history

Status must be success or insufficient_data.

Empty findings are allowed.

Patient ID:
{patient_id}

PROVIDED EVIDENCE:
{json.dumps(
    evidence,
    ensure_ascii=False,
    allow_nan=False
)}

Required shape:
{json.dumps(
    {
        "patient_id": patient_id,
        "status": "success",
        "findings": []
    }
)}
"""


    try:

        output = parse_agent_json(
            ask_iaedu(prompt),
            ClinicalAgentOutput
        )


        validated_selection(
            output,
            patient_id,
            evidence
        )


        state.agent_output = output

        state.status = output.status


    except (
        requests.RequestException,
        ValidationError,
        ValueError,
        RuntimeError
    ) as exc:

        # Guardar dados determinísticos para o
        # Orchestrator, mas nunca aceitar saída
        # LLM inválida.

        state.status = "error"

        state.error_message = (
            f"{type(exc).__name__}: "
            f"{str(exc)[:350]}"
        )

        state.agent_output = (
            ClinicalAgentOutput(
                patient_id=patient_id,
                status="error",
                findings=[]
            )
        )


    return state

### 7. Teste local (não requer IAedu) - dados, regras e padrões

In [9]:
TEST_PATIENT = 153972

patient = get_clinical_data(TEST_PATIENT)

if "error" in patient:
    print("Paciente de teste não encontrado neste CSV. Escolhe um ID existente:")
    print(clinical_df.index[:5].tolist())

else:
    evidence = build_clinical_evidence(patient)
    print("Dados clínicos:")
    print(json.dumps(patient,indent=2,ensure_ascii=False))
    print("\nEvidência disponível:",len(evidence))
    print("Exemplos de evidência:")
    print(json.dumps(evidence[:10],indent=2,ensure_ascii=False))

Dados clínicos:
{
  "patientunitstayid": 153972,
  "age": 63.0,
  "gender": "Male",
  "unit_type": "Med-Surg ICU",
  "admission_source": "Emergency Department",
  "admission_diagnosis": "Bleeding, lower GI",
  "diagnoses_0_24h": [
    "cardiovascular|shock / hypotension|hypotension",
    "cardiovascular|chest pain / ASHD|coronary artery disease|s/p CABG",
    "cardiovascular|arrhythmias|atrial fibrillation",
    "pulmonary|disorders of the airways|obstructive sleep apnea",
    "hematology|bleeding and red blood cell disorders|anemia",
    "cardiovascular|ventricular disorders|cardiomyopathy|dilated|ischemic",
    "gastrointestinal|GI bleeding / PUD|GI bleeding",
    "hematology|coagulation disorders|coagulopathy|coumadin administration"
  ],
  "past_history": [
    "Cardiovascular: Coronary Artery Bypass - CABG - remote",
    "Cardiovascular: Arrhythmias - atrial fibrillation - chronic",
    "Cardiovascular: Hypertension Requiring Treatment",
    "Cardiovascular: AICD",
    "Cardiovasc

### 8. Teste de paciente inexistente - não chama IAedu

In [10]:
missing = run_clinical_agent(
    999999999
)


assert missing.status == "error"


assert (
    missing.agent_output.status
    == "error"
)


print(
    missing.model_dump_json(
        indent=2
    )
)

{
  "patient_id": 999999999,
  "status": "error",
  "raw_data": {},
  "evidence": [],
  "agent_output": {
    "patient_id": 999999999,
    "status": "error",
    "findings": []
  },
  "error_message": "Patient 999999999 not found."
}


### 9. Teste completo (faz uma chamada real à IAedu)

In [11]:
state = run_clinical_agent(TEST_PATIENT)

print("Estado:", state.status)

print(
    "Agent output:",
    state.agent_output.model_dump_json(indent=2)
    if state.agent_output
    else None
)

print("Evidência:", len(state.evidence))

if state.error_message:
    print("Erro:", state.error_message)

Estado: success
Agent output: {
  "patient_id": 153972,
  "status": "success",
  "findings": [
    {
      "category": "age",
      "value": "63"
    },
    {
      "category": "gender",
      "value": "Male"
    },
    {
      "category": "unit_type",
      "value": "Med-Surg ICU"
    },
    {
      "category": "admission_source",
      "value": "Emergency Department"
    },
    {
      "category": "admission_diagnosis",
      "value": "Bleeding, lower GI"
    },
    {
      "category": "diagnosis_0_24h",
      "value": "cardiovascular|shock / hypotension|hypotension"
    },
    {
      "category": "diagnosis_0_24h",
      "value": "cardiovascular|chest pain / ASHD|coronary artery disease|s/p CABG"
    },
    {
      "category": "diagnosis_0_24h",
      "value": "cardiovascular|arrhythmias|atrial fibrillation"
    },
    {
      "category": "diagnosis_0_24h",
      "value": "hematology|bleeding and red blood cell disorders|anemia"
    },
    {
      "category": "diagnosis_0_24h",
    